In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
GLOB_DIR = Path(cfg.dataPath) / path_cache
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make sure the base directory exists
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
# SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'CEU']
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'ALA']
experiment_region_groups = {"CEU": ["FR", "IT_AT", "CH"]}
res_xreg_by_source = {}
split_info_by_source = {}

rerun_src_regions = {}
for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Source region: {src_region}")
    if src_region in rerun_src_regions:
        RUN_FLAG = True
    else:
        RUN_FLAG = False
    res_xreg, split_info = prepare_monthly_df_xreg_SOURCE_to_EU(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        run_flag=RUN_FLAG,
        source_code=src_region,
        region_groups=experiment_region_groups)

    res_xreg_by_source[src_region] = res_xreg
    split_info_by_source[src_region] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    print(f"Train glaciers ({src_region}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers (non-{src_region}):",
          len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "Test rows:", len(df_test))

In [ ]:
# EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}  # set to empty set to include all
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH": ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "ALA": ["ALA"],
}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = ["CH", "NOR", "ISL", "ALA", "SJM"]

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_members]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']
# define scalars:
scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
def split_pool_holdout_sinkhorn(
    df_region: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    scaler_m,
    scaler_s,
    blur_joint: float,
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    elev_diff_col: str = "ELEVATION_DIFFERENCE",
    pool_frac: float = 0.2,
    seed: int = 0,
    n_restarts: int = 50,
    frac_tol: float = 0.05,
    min_pool_glaciers: int = 3,
) -> dict:
    """
    Split a target region into a fine-tuning pool and a holdout set.

    Distances are computed at measurement-row level using the true joint
    (climate + topo stacked) Sinkhorn divergence, consistent with
    D_sinkhorn_joint in compute_domain_shift.

    The split is optimized so that:
      (1) Sinkhorn(pool → holdout) is small  — pool covers the holdout distribution
      (2) |Sinkhorn(pool → all) - Sinkhorn(holdout → all)| is small — balanced shift

    Parameters
    ----------
    df_region : pd.DataFrame
        All rows for the target region.
    monthly_cols : list[str]
        Monthly feature column names.
    static_cols : list[str]
        Static feature column names.
    scaler_m :
        Global monthly scaler fitted via build_global_scalers_multi_source.
    scaler_s :
        Global static scaler fitted via build_global_scalers_multi_source.
    blur_joint : float
        Sinkhorn blur for the joint space — pass blur_joint from 1.7.
    glacier_col : str
        Column identifying glaciers.
    id_col : str
        Column identifying individual measurements.
    elev_diff_col : str
        Name of the elevation difference column (averaged per ID, not .first()).
    pool_frac : float
        Target fraction of measurements to use as fine-tuning pool.
    seed : int
        Base random seed.
    n_restarts : int
        Number of random orderings to try; best split by score is kept.
    frac_tol : float
        Splits where |achieved_frac - pool_frac| > frac_tol are skipped.
    min_pool_glaciers : int
        Minimum number of glaciers required in the pool.

    Returns
    -------
    dict with keys:
        pool_glaciers, holdout_glaciers,
        n_meas_pool, n_meas_holdout,
        actual_pool_frac,
        sinkhorn_pool_vs_holdout,
        sinkhorn_pool_vs_region,
        sinkhorn_holdout_vs_region,
        blur_joint, best_seed, best_score
    """
    # ------------------------------------------------------------------
    # 1. Glacier inventory — names and measurement counts
    # ------------------------------------------------------------------
    glaciers = df_region[glacier_col].unique().tolist()
    n_glaciers = len(glaciers)
    glacier_to_idx = {g: i for i, g in enumerate(glaciers)}

    n_meas = np.array([(df_region[glacier_col] == g).sum() for g in glaciers],
                      dtype=int)

    n_total_meas = int(n_meas.sum())
    target_pool_meas = int(round(pool_frac * n_total_meas))

    print(f"  Total measurements : {n_total_meas}")
    print(f"  Target pool meas   : {target_pool_meas} ({pool_frac:.0%})")
    print(f"  Total glaciers     : {n_glaciers}")
    print(f"  blur_joint         : {blur_joint:.4f}")

    # ------------------------------------------------------------------
    # 2. Precompute row-level joint features for the whole region
    #    (consistent with compute_domain_shift's D_sinkhorn_joint)
    # ------------------------------------------------------------------
    pure_static = [c for c in static_cols if c != elev_diff_col]

    def _stake_topo(df):
        parts = [df.groupby(id_col)[pure_static].first()]
        if elev_diff_col in static_cols:
            parts.append(df.groupby(id_col)[[elev_diff_col]].mean())
        return pd.concat(parts, axis=1)[static_cols].to_numpy(dtype=np.float64)

    Xm_all = scaler_m.transform(
        df_region[monthly_cols].to_numpy(dtype=np.float64))
    Xs_id = scaler_s.transform(_stake_topo(df_region))
    id_codes = pd.Categorical(df_region[id_col]).codes
    topo_per_row = Xs_id[id_codes]
    X_joint_all = np.hstack([Xm_all, topo_per_row]).astype(np.float32)

    # row-to-glacier mapping for fast subsetting
    glacier_codes = np.array(
        [glacier_to_idx[g] for g in df_region[glacier_col]], dtype=int)

    loss_fn = SamplesLoss(
        loss="sinkhorn",
        p=2,
        blur=blur_joint,
        scaling=0.9,
        debias=True,
        backend="tensorized",
    )

    def _sinkhorn(mask_a, mask_b, max_samples=2000):
        Xa = X_joint_all[mask_a]
        Xb = X_joint_all[mask_b]
        if len(Xa) < 2 or len(Xb) < 2:
            return 0.0
        if len(Xa) > max_samples:
            Xa = Xa[np.random.choice(len(Xa), max_samples, replace=False)]
        if len(Xb) > max_samples:
            Xb = Xb[np.random.choice(len(Xb), max_samples, replace=False)]
        ta = torch.as_tensor(Xa, dtype=torch.float32)
        tb = torch.as_tensor(Xb, dtype=torch.float32)
        wa = torch.ones(len(ta)) / len(ta)
        wb = torch.ones(len(tb)) / len(tb)
        with torch.no_grad():
            return float(loss_fn(wa, ta, wb, tb).item())

    def _mask_for(glacier_idxs):
        return np.isin(glacier_codes, list(glacier_idxs))

    all_mask = np.ones(len(X_joint_all), dtype=bool)

    # ------------------------------------------------------------------
    # 3. Greedy split with n_restarts random orderings, keep best score
    # ------------------------------------------------------------------
    best_score = np.inf
    best_result = None

    for restart in range(n_restarts):
        rng = np.random.default_rng(seed + restart)
        order = np.arange(n_glaciers)
        rng.shuffle(order)

        pool_idxs = set()
        holdout_idxs = set()
        pool_meas_count = 0

        for glacier_idx in order:
            gl_meas = int(n_meas[glacier_idx])
            pool_full = pool_meas_count + gl_meas > target_pool_meas * (
                1 + frac_tol)

            if pool_full:
                holdout_idxs.add(glacier_idx)
                continue

            trial_pool = pool_idxs | {glacier_idx}
            trial_holdout = holdout_idxs | {glacier_idx}

            holdout_mask = _mask_for(
                holdout_idxs) if holdout_idxs else all_mask
            pool_mask = _mask_for(pool_idxs) if pool_idxs else all_mask

            score_if_pool = (_sinkhorn(_mask_for(trial_pool), holdout_mask) +
                             abs(
                                 _sinkhorn(_mask_for(trial_pool), all_mask) -
                                 _sinkhorn(holdout_mask, all_mask)))
            score_if_holdout = (
                _sinkhorn(pool_mask, _mask_for(trial_holdout)) + abs(
                    _sinkhorn(pool_mask, all_mask) -
                    _sinkhorn(_mask_for(trial_holdout), all_mask)))

            if score_if_pool <= score_if_holdout:
                pool_idxs.add(glacier_idx)
                pool_meas_count += gl_meas
            else:
                holdout_idxs.add(glacier_idx)

        achieved_frac = pool_meas_count / n_total_meas
        if abs(achieved_frac - pool_frac) > frac_tol:
            continue
        if len(pool_idxs) < min_pool_glaciers:
            continue

        pool_mask = _mask_for(pool_idxs)
        holdout_mask = _mask_for(holdout_idxs)

        sk_pool_holdout = _sinkhorn(pool_mask, holdout_mask)
        sk_pool_region = _sinkhorn(pool_mask, all_mask)
        sk_holdout_region = _sinkhorn(holdout_mask, all_mask)

        score = sk_pool_holdout + abs(sk_pool_region - sk_holdout_region)

        print(f"  restart {restart:3d} | frac={achieved_frac:.2f} | "
              f"sk(pool↔holdout)={sk_pool_holdout:.4f} | "
              f"balance={abs(sk_pool_region - sk_holdout_region):.4f} | "
              f"score={score:.4f}")

        if score < best_score:
            best_score = score
            best_result = {
                "pool_idxs": pool_idxs,
                "holdout_idxs": holdout_idxs,
                "pool_meas_count": pool_meas_count,
                "achieved_frac": achieved_frac,
                "sk_pool_holdout": sk_pool_holdout,
                "sk_pool_region": sk_pool_region,
                "sk_holdout_region": sk_holdout_region,
                "best_seed": seed + restart,
            }

    if best_result is None:
        raise RuntimeError(
            f"No valid split found within frac_tol={frac_tol} and "
            f"min_pool_glaciers={min_pool_glaciers} after {n_restarts} restarts. "
            f"Try increasing n_restarts or frac_tol.")

    r = best_result
    pool_glaciers = [glaciers[i] for i in r["pool_idxs"]]
    holdout_glaciers = [glaciers[i] for i in r["holdout_idxs"]]

    print(f"\n  Best split (seed={r['best_seed']}):")
    print(
        f"  Pool    : {len(pool_glaciers)} glaciers, {r['pool_meas_count']} meas ({r['achieved_frac']:.1%})"
    )
    print(
        f"  Holdout : {len(holdout_glaciers)} glaciers, {n_total_meas - r['pool_meas_count']} meas ({1 - r['achieved_frac']:.1%})"
    )
    print(f"  Sinkhorn(pool → holdout)  : {r['sk_pool_holdout']:.4f}")
    print(f"  Sinkhorn(pool → region)   : {r['sk_pool_region']:.4f}")
    print(f"  Sinkhorn(holdout → region): {r['sk_holdout_region']:.4f}")
    print(f"  Score                     : {best_score:.4f}")

    return {
        "pool_glaciers": pool_glaciers,
        "holdout_glaciers": holdout_glaciers,
        "n_meas_pool": int(r["pool_meas_count"]),
        "n_meas_holdout": int(n_total_meas - r["pool_meas_count"]),
        "actual_pool_frac": float(r["achieved_frac"]),
        "sinkhorn_pool_vs_holdout": float(r["sk_pool_holdout"]),
        "sinkhorn_pool_vs_region": float(r["sk_pool_region"]),
        "sinkhorn_holdout_vs_region": float(r["sk_holdout_region"]),
        "blur_joint": float(blur_joint),
        "best_seed": int(r["best_seed"]),
        "best_score": float(best_score),
    }

In [ ]:
RUN = False
RECOMPUTE_SPLITS = False  # set to True to rerun the optimization
save_path = BASE_DIR / "glacier_splits.json"

if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        output = json.load(f)
    splits = {
        src_region: {
            "pool_glaciers": data["pool_glaciers"],
            "holdout_glaciers": data["holdout_glaciers"],
            "actual_pool_frac": data["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": data["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": data["sinkhorn_holdout_vs_region"],
            "blur_joint": data["blur_joint"],
            "best_seed": data["best_seed"],
            "best_score": data["best_score"],
        }
        for src_region, data in output.items()
    }
    print(f"Loaded splits from: {save_path}")
    for src_region, split in splits.items():
        print(f"\n{src_region}:")
        print(f"  Pool glaciers     : {split['pool_glaciers']}")
        print(f"  Holdout glaciers  : {split['holdout_glaciers']}")
        print(f"  Pool fraction     : {split['actual_pool_frac']:.1%}")
        print(f"  Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")

else:
    splits = {}

    for src_region in SOURCE_REGIONS:
        print(f"\n{'='*50}")
        print(f"Splitting region: {src_region}")

        df_target = res_xreg_by_source[src_region]['df_train']

        split = split_pool_holdout_sinkhorn(
            df_region=df_target,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            pool_frac=0.2,
            n_restarts=50,
            seed=cfg.seed,
        )

        splits[src_region] = split

        print(f"\nPool glaciers     : {split['pool_glaciers']}")
        print(f"Holdout glaciers  : {split['holdout_glaciers']}")
        print(f"Pool fraction     : {split['actual_pool_frac']:.1%}")
        print(f"Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")
        print(f"Sk(pool↔region)   : {split['sinkhorn_pool_vs_region']:.4f}")
        print(f"Sk(holdout↔region): {split['sinkhorn_holdout_vs_region']:.4f}")

    output = {
        src_region: {
            "pool_glaciers": split["pool_glaciers"],
            "holdout_glaciers": split["holdout_glaciers"],
            "actual_pool_frac": split["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": split["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": split["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": split["sinkhorn_holdout_vs_region"],
            "blur_joint": split["blur_joint"],
            "best_seed": split["best_seed"],
            "best_score": split["best_score"],
        }
        for src_region, split in splits.items()
    }

    with open(save_path, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\nAll splits saved to: {save_path}")


#### Final ft and hold-out glaciers:

In [ ]:
# df_ch_train = res_xreg_by_source['CH']['df_train']
# df_ch_test = res_xreg_by_source['CH']['df_test']

# df_ceu_train = res_xreg_by_source['CEU']['df_train']
# df_ceu_train_filtered = df_ceu_train[df_ceu_train['SOURCE_CODE'].isin(
#     ['FR', 'IT_AT', 'CH'])].copy()
# df_ceu_train_filtered['SOURCE_CODE'] = 'CEU'

# # combine all relevant data for verification
# df_ch_all = pd.concat([df_ch_train, df_ch_test])

# # CEU rows need to come from the filtered train set (relabeled as CEU)
# df_verify = pd.concat([df_ch_all, df_ceu_train_filtered])

# df_row_check = verify_row_percentage(
#     df_verify,
#     FT_GLACIERS,
#     region_groups={"CEU": ["CEU"]},  # already relabeled, so just match "CEU"
# )
# df_row_check

In [ ]:
FT_GLACIERS = {
    src_region: {
        "20pct": splits[src_region]["pool_glaciers"],
    }
    for src_region in SOURCE_REGIONS
}
df_row_check = verify_row_percentage(
    pd.concat([
        res_xreg_by_source[src_region]['df_train']
        for src_region in SOURCE_REGIONS
    ]),
    FT_GLACIERS,
)
df_row_check

### Plot FT/Hold-out glaciers:

##### Switzerland:

In [ ]:
ceu_pool = FT_GLACIERS["CH"]["20pct"]
df_ceu_all = res_xreg_by_source['CH']['df_train']

ft_glaciers_by_split = {
    "20pct": ceu_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_ceu_all.loc[~df_ceu_all["GLACIER"].isin(ceu_pool),
                   "GLACIER"].unique().tolist(),
}

data_CEU, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="11",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_CEU_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b", "Excluded": "#878787"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_CEU_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central European Alps (20pct)",
    extent=(5.8, 15, 44, 48),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

##### Norway:

In [ ]:
nor_pool = FT_GLACIERS["NOR"]["20pct"]
df_nor_all = res_xreg_by_source['NOR']['df_train']

ft_glaciers_by_split = {
    "20pct": nor_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_nor_all.loc[~df_nor_all["GLACIER"].isin(nor_pool),
                   "GLACIER"].unique().tolist(),
}

data_NOR, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="08",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_NOR_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_NOR_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Norway (20pct)",
    extent=(4, 24, 57, 71),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
    legend_ncol=2,
)

##### Alaska:

In [ ]:
ala_pool = FT_GLACIERS["ALA"]["20pct"]
df_ala_all = res_xreg_by_source['ALA']['df_train']

ft_glaciers_by_split = {
    "20pct": ala_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_ala_all.loc[~df_ala_all["GLACIER"].isin(ala_pool),
                   "GLACIER"].unique().tolist(),
}

data_ala, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="01",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ala_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ala_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Alaska (20pct)",
    extent=(-170, -130, 55, 70),
    sizes=(100, 1000),
    size_legend_values=(30, 100, 400),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

##### Iceland:

In [ ]:
isl_pool = FT_GLACIERS["ISL"]["20pct"]
df_isl_all = res_xreg_by_source['ISL']['df_train']

ft_glaciers_by_split = {
    "20pct": isl_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_isl_all.loc[~df_isl_all["GLACIER"].isin(isl_pool),
                   "GLACIER"].unique().tolist(),
}

data_ISL, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ISL_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Iceland (20pct)",
    extent=(-25, -11, 62, 68),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

### Plot feature overlap:

In [ ]:
# # res_xreg is the ONE dict from your cross-regional monthly prep
# figs_by_code = plot_tsne_overlap_xreg_from_single_res(
#     res_xreg=res_xreg,
#     cfg=cfg,
#     STATIC_COLS=STATIC_COLS,
#     MONTHLY_COLS=MONTHLY_COLS,
#     group_col="SOURCE_CODE",
#     ch_code="CH",
#     use_aug=False,  # or True if you want *_aug
#     n_iter=1000,
#     # only_codes=["IT_AT"],  # optional
# )

In [ ]:
# FEATURES = MONTHLY_COLS + STATIC_COLS + ["POINT_BALANCE"]

# figs_kde = plot_feature_kde_overlap_xreg_ch_vs_codes(
#     res_xreg=res_xreg,
#     cfg=cfg,
#     features=FEATURES,
#     group_col="SOURCE_CODE",
#     ch_code="CH",
#     use_aug=True,  # usually best for feature overlap
#     # only_codes=["IT_AT", "FR"],    # optional
#     output_dir="figures/xreg_kde",  # optional
# )


## LSTM model
### LSTM datasets:

In [ ]:
tl_assets_by_source = {}

# REGION_GROUPS = {"CEU": ["FR", "IT_AT", "CH"]}
REGION_GROUPS = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Building TL assets for source: {src_region}")

    # Skip target regions that overlap with the source
    src_members = set(REGION_GROUPS.get(src_region, [src_region]))
    ft_glaciers_for_src = {
        reg: splits
        for reg, splits in FT_GLACIERS.items()
        if reg not in src_members and not src_members.issubset(
            REGION_GROUPS.get(reg, [reg])) and reg != src_region
    }

    tl_assets_by_source[src_region] = build_transfer_learning_assets(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        FT_GLACIERS=ft_glaciers_for_src,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=BASE_DIR / "LSTM_cache_TL" / src_region,
        force_recompute=False,
        src_code=src_region,
        region_groups=REGION_GROUPS,
    )
    print(f"{src_region} -> TL asset keys:",
          list(tl_assets_by_source[src_region].keys()))

### LSTM parameters:

In [ ]:
log_path_gs_results = {
    "ISL":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "FR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_FR_2026-02-06.csv',
    "CH": GLOB_DIR / 'GS_results/lstm_param_search_progress_CH_2026-02-18.csv',
}

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)


def _params_key_for_source(src_region, params_by_key, default_params):
    """Look up best params for a source region using RGI_REGIONS mapping."""
    for rgi, info in RGI_REGIONS.items():
        if info["code"] == src_region:
            key = f"{rgi}_{src_region}"
            return params_by_key.get(key, default_params)
    # CEU has no single RGI — fall back to default
    return default_params


params_by_key.keys()

### Train or load CH baseline:

In [ ]:
baseline_models = {}
baseline_paths = {}

TRAIN_FLAG = False
MODEL_DATE = "2026-04-28"

for src_region in SOURCE_REGIONS:
    # pick any key from this source's tl_assets to get the pretrain dataset
    any_key = next(iter(tl_assets_by_source[src_region]))

    model, path, info = train_or_load_CH_baseline(
        cfg=cfg,
        tl_assets=tl_assets_by_source[src_region],
        default_params=_params_key_for_source(src_region, params_by_key,
                                              default_params),
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_{src_region}",
        key="defaultparams",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        verbose=False,
        date=MODEL_DATE,
    )

    baseline_models[src_region] = model
    baseline_paths[src_region] = path
    print(f"{src_region} baseline saved at: {path}")

### Fine tune LSTM:

In [ ]:
models_tl_ft_simple_by_source = {}
infos_tl_ft_simple_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Fine-tuning from source: {src_region}")

    models_tl_ft_simple_by_source[src_region], \
    infos_tl_ft_simple_by_source[src_region] = finetune_TL_models_all(
        cfg=cfg,
        tl_assets_by_key=tl_assets_by_source[src_region],
        best_params=_params_key_for_source(src_region, params_by_key, default_params),
        device=device,
        pretrained_ckpt_path=baseline_paths[src_region],
        strategies=("safe", "full", "disc_full"),
        force_retrain=False,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_{src_region}",
        verbose=False,
        best_by_region=None,
        date="2026-04-28",
    )

    print(f"{src_region} -> fine-tuned model keys:",
          list(models_tl_ft_simple_by_source[src_region].keys()))

### Adapters:

In [ ]:
# load GS results:
import json

with open(
        GLOB_DIR /
        'GS_results/tuning_adapter/adapter_best_by_region_2026-02-24_15.json',
        "r") as f:
    best_by_region = json.load(f)

best_by_region

In [ ]:
models_tl_ft_adapter_by_source = {}
infos_tl_ft_adapter_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Fine-tuning adapters from source: {src_region}")

    models_tl_ft_adapter_by_source[src_region], \
    infos_tl_ft_adapter_by_source[src_region] = finetune_TL_models_all(
        cfg=cfg,
        tl_assets_by_key=tl_assets_by_source[src_region],
        best_params=_params_key_for_source(src_region, params_by_key, default_params),
        device=device,
        pretrained_ckpt_path=baseline_paths[src_region],
        strategies=["adapter"],
        force_retrain=False,
        prefix=f"lstm_{src_region}",
        verbose=False,
        best_by_region=best_by_region,
        models_dir=BASE_DIR / "models" / src_region,
        date="2026-04-28",
    )

    print(f"{src_region} -> adapter model keys:",
          list(models_tl_ft_adapter_by_source[src_region].keys()))

### DAN:

In [ ]:
models_dan_by_source = {}
infos_dan_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Fine-tuning DAN from source: {src_region}")

    # DAN only makes sense for high-shift targets
    dan_regions = [
        k.split(f"TL_{src_region}_to_")[1].rsplit("_", 1)[0]
        for k in tl_assets_by_source[src_region].keys()
    ]
    dan_regions = sorted(set(dan_regions))

    models_dan_by_source[src_region], \
    infos_dan_by_source[src_region] = finetune_TL_models_all(
        cfg=cfg,
        tl_assets_by_key=tl_assets_by_source[src_region],
        best_params=_params_key_for_source(src_region, params_by_key, default_params),
        device=device,
        pretrained_ckpt_path=baseline_paths[src_region],
        strategies=["dan"],
        force_retrain=False,
        prefix=f"lstm_{src_region}",
        verbose=False,
        regions_only=dan_regions,
        best_by_region=best_by_region,
        models_dir=BASE_DIR / "models" / src_region,
        date="2026-04-30",
    )

    print(f"{src_region} -> DAN model keys:",
          list(models_dan_by_source[src_region].keys()))

### Evaluate on test:

In [ ]:
# baseline_models and baseline_paths are already loaded above
# via train_or_load_CH_baseline loop — no need to reload

# Sanity check
for src_region in SOURCE_REGIONS:
    print(f"{src_region} -> baseline loaded: {baseline_paths[src_region]}")

##### 20 percent split (sparse):

In [ ]:
tl_assets_by_source['CH'].keys()

In [ ]:
ax = plt.subplot(1, 1, 1)
src_region = 'CH'
target_region = 'ISL'

prefix = f"TL_{src_region}_to_"
target_regions = sorted(
    set(k[len(prefix):].rsplit("_", 1)[0]
        for k in tl_assets_by_source[src_region].keys()))
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}
metrics, df_preds, _fig_ind, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=models_xreg_for_src[target_region],
    device=device,
    tl_assets_for_key=tl_assets_by_source[src_region]
    [f'TL_{src_region}_to_{target_region}_20pct'],
    ax=ax,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False)

In [ ]:
ax = plt.subplot(1, 1, 1)
src_region = 'CH'
target_region = 'ISL'

prefix = f"TL_{src_region}_to_"
target_regions = sorted(
    set(k[len(prefix):].rsplit("_", 1)[0]
        for k in tl_assets_by_source[src_region].keys()))
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}
models_total = {
    **models_tl_ft_simple_by_source[src_region],
    **models_tl_ft_adapter_by_source[src_region],
    **models_dan_by_source[src_region],
}
metrics, df_preds, _fig_ind, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=models_total[f'TL_{src_region}_to_{target_region}_20pct__dan'],
    device=device,
    tl_assets_for_key=tl_assets_by_source[src_region]
    [f'TL_{src_region}_to_{target_region}_20pct'],
    ax=ax,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False)

In [ ]:
src_region = 'CH'
split_name = '20pct'

models_total = {
    **models_tl_ft_simple_by_source[src_region],
    **models_tl_ft_adapter_by_source[src_region],
    **models_dan_by_source[src_region],
}

prefix = f"TL_{src_region}_to_"
# target_regions = sorted(
#     set(k[len(prefix):].rsplit("_", 1)[0]
#         for k in tl_assets_by_source[src_region].keys()))
target_regions = ['ISL']
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}

df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
    cfg=cfg,
    regions=target_regions,
    models_xreg_by_region=models_xreg_for_src,
    models_tl_by_key=models_total,
    tl_assets_by_key=tl_assets_by_source[src_region],
    device=device,
    split_name=split_name,
    src_code=src_region,
    save_dir=f"figures/eval_TL/{src_region}",
    strategies=["no_ft", "adapter", "dan"],
    strategy_labels={
        "no_ft": "No FT",
        "adapter": "Adapter FT",
        "dan": "Domain-Adversarial",
    },
)

plt.show()

In [ ]:
src_region = 'ISL'
split_name = '20pct'

models_total = {
    **models_tl_ft_simple_by_source[src_region],
    **models_tl_ft_adapter_by_source[src_region],
    **models_dan_by_source[src_region],
}

prefix = f"TL_{src_region}_to_"
# target_regions = sorted(
#     set(k[len(prefix):].rsplit("_", 1)[0]
#         for k in tl_assets_by_source[src_region].keys()))
target_regions = ['CH']
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}

df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
    cfg=cfg,
    regions=target_regions,
    models_xreg_by_region=models_xreg_for_src,
    models_tl_by_key=models_total,
    tl_assets_by_key=tl_assets_by_source[src_region],
    device=device,
    split_name=split_name,
    src_code=src_region,
    save_dir=f"figures/eval_TL/{src_region}",
    strategies=["no_ft", "adapter", "dan"],
    strategy_labels={
        "no_ft": "No FT",
        "adapter": "Adapter FT",
        "dan": "Domain-Adversarial",
    },
)

plt.show()

In [ ]:
df_metrics_tl_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Evaluating source: {src_region}")

    models_total = {
        **models_tl_ft_simple_by_source[src_region],
        **models_tl_ft_adapter_by_source[src_region],
        **models_dan_by_source[src_region],
    }

    prefix = f"TL_{src_region}_to_"
    target_regions = sorted(
        set(k[len(prefix):].rsplit("_", 1)[0]
            for k in tl_assets_by_source[src_region].keys()))
    print(f"Target regions: {target_regions}")

    models_xreg_for_src = {
        reg: baseline_models[src_region]
        for reg in target_regions
    }

    df_metrics_tl_by_source[src_region] = {}

    for split_name in ["20pct"]:
        df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
            cfg=cfg,
            regions=target_regions,
            models_xreg_by_region=models_xreg_for_src,
            models_tl_by_key=models_total,
            tl_assets_by_key=tl_assets_by_source[src_region],
            device=device,
            split_name=split_name,
            src_code=src_region,
            save_dir=f"figures/eval_TL/{src_region}",
            strategies=["no_ft", "safe", "full", "adapter", "dan"],
        )

        df_metrics_tl_by_source[src_region][split_name] = df_tl_grid
        plt.show()

### Compare against distance:

In [ ]:
import pickle

RECOMPUTE_SHIFTS = False

CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

EXCLUDE_TARGETS = {}
SPLIT = "20pct"  # use same split as the evaluation

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache_tl.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    # Scalers and bandwidths fitted on train data only (same as 1.7)
    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    # Compute shifts using holdout test set from tl_assets
    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        df_src_train = res_xreg_by_source[src_region]["df_train"]

        for exp_key, assets in tl_assets_by_source[src_region].items():
            if not exp_key.endswith(f"_{SPLIT}"):
                continue

            tgt = exp_key.split(f"TL_{src_region}_to_")[1].rsplit(
                f"_{SPLIT}", 1)[0]
            if tgt.upper() in EXCLUDE_TARGETS:
                continue

            ds_test = assets.get("ds_test")
            if ds_test is None:
                continue

            shift_key = f"XREG_{src_region}_TO_{tgt}"
            ft_glaciers = assets["ft_glaciers"]

            # expand group codes for CEU
            tgt_codes = REGION_GROUPS.get(tgt, [tgt])

            df_tgt_all = res_xreg_by_source[src_region]["df_test"]

            # for train-side targets (CH within CEU source), also check df_train
            df_tgt_train = res_xreg_by_source[src_region]["df_train"]
            df_tgt_pool = pd.concat([
                df_tgt_all[df_tgt_all["SOURCE_CODE"].isin(tgt_codes)],
                df_tgt_train[df_tgt_train["SOURCE_CODE"].isin(tgt_codes)],
            ]).drop_duplicates()

            df_tgt_holdout = df_tgt_pool[~df_tgt_pool["GLACIER"].
                                         isin(ft_glaciers)]

            if len(df_tgt_holdout) == 0:
                print(f"Warning: empty holdout for {shift_key}, skipping")
                continue

            shift = compute_domain_shift(
                df_src=df_src_train,
                df_tgt=df_tgt_holdout,
                monthly_cols=CLIMATE_COLS,
                static_cols=TOPO_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                seed=cfg.seed,
            )
            all_shifts[shift_key] = shift
            print(f"  {shift_key}: {len(df_tgt_holdout)} holdout rows")

            all_shifts_by_source[src_region] = all_shifts
            print(f"{src_region} -> computed {len(all_shifts)} shifts")

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

In [ ]:
SPLIT = "20pct"
EXCLUDE_TARGETS = []


def _extract_metrics_for_strategy(strategy):
    rows = []
    for src_region in SOURCE_REGIONS:
        if src_region not in df_metrics_tl_by_source:
            continue
        df_src = df_metrics_tl_by_source[src_region][SPLIT]

        # filter to this strategy
        df_strat = df_src.xs(strategy, level="strategy")

        for region, row in df_strat.iterrows():
            shift_key = f"XREG_{src_region}_TO_{region}"
            if shift_key not in all_shifts_by_source.get(src_region, {}):
                print(f"Warning: {shift_key} not in shifts, skipping")
                continue
            rows.append({
                **row.to_dict(),
                "key": shift_key,
                "src": src_region,
                "tgt": region,
            })
    return pd.DataFrame(rows).set_index("key") if rows else pd.DataFrame()


df_metrics_zs_all = _extract_metrics_for_strategy("no_ft")
df_metrics_ft_all = _extract_metrics_for_strategy("adapter")

# --- combine shifts ---
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source.get(src_region, {}))

# --- apply exclusions ---
if EXCLUDE_TARGETS:
    excl_pattern = "|".join(EXCLUDE_TARGETS)
    df_metrics_zs_all = df_metrics_zs_all[~df_metrics_zs_all.index.str.
                                          contains(excl_pattern)]
    df_metrics_ft_all = df_metrics_ft_all[~df_metrics_ft_all.index.str.
                                          contains(excl_pattern)]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

print(
    f"Zero-shot metrics: {df_metrics_zs_all.shape} — {list(df_metrics_zs_all.index)}"
)
print(
    f"FT metrics:        {df_metrics_ft_all.shape} — {list(df_metrics_ft_all.index)}"
)

In [ ]:
EXCLUDE_TARGETS = []
color_palette = {
    'CH': NATURE_PALETTE["blue"],
    'NOR': NATURE_PALETTE["vermillion"],
    'ALA': NATURE_PALETTE["bluish_green"],
    'ISL': NATURE_PALETTE["reddish_purple"]
}
# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_zs_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_zs_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_zs_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
    color_palette=color_palette)

In [ ]:
EXCLUDE_TARGETS = []

# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_ft_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_ft_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_ft_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
    color_palette=color_palette)

In [ ]:
def plot_region_shift_vs_performance_multi(
    datasets: list[tuple[pd.DataFrame, str]],  # [(df_metrics, title), ...]
    all_shifts: dict,
    complement_key: str = "",
    performance_cols: list[str] | None = None,
    distance_variant: str = "mmd2",
    exclude_targets: list[str] | None = None,
    exclude_sources: list[str] | None = None,
    blur_m: float | None = None,
    blur_s: float | None = None,
    blur_joint: float | None = None,
    joint_variant: str = "averaged",
    distance_cols_override: list[str] | None = None,
    figsize: tuple | None = None,
    sharey: bool = True,
    color_palette: dict | None = None,
):
    from scipy import stats

    if joint_variant not in {"averaged", "true"}:
        raise ValueError("joint_variant must be 'averaged' or 'true'.")

    exclude_targets = {t.upper() for t in (exclude_targets or [])}
    exclude_sources = {s.upper() for s in (exclude_sources or [])}

    if joint_variant == "true" and distance_variant == "sinkhorn":
        joint_col = "D_sinkhorn_joint"
        joint_label = "sinkhorn joint (true)"
    else:
        if joint_variant == "true":
            print(
                f"Warning: joint_variant='true' only available for sinkhorn, "
                f"falling back to 'averaged' for {distance_variant}.")
        joint_col = f"D_{distance_variant}_joint"
        joint_label = f"{distance_variant} joint (averaged)"

    all_distance_cols = [
        joint_col,
        f"D_{distance_variant}_climate",
        f"D_{distance_variant}_topo",
    ]
    all_xlabel_map = {
        joint_col: joint_label,
        f"D_{distance_variant}_climate": f"{distance_variant} climate",
        f"D_{distance_variant}_topo": f"{distance_variant} topo",
    }
    distance_cols = distance_cols_override if distance_cols_override is not None else all_distance_cols
    xlabel_map = {
        k: v
        for k, v in all_xlabel_map.items() if k in distance_cols
    }

    blur_map = {}
    if distance_variant == "sinkhorn":
        if blur_m is not None and blur_s is not None:
            blur_map["D_sinkhorn_joint"] = 0.5 * (blur_m + blur_s)
            blur_map["D_sinkhorn_climate"] = blur_m
            blur_map["D_sinkhorn_topo"] = blur_s
        if blur_joint is not None:
            blur_map["D_sinkhorn_joint"] = blur_joint

    # --- resolve performance_cols from first dataset if not provided ---
    if performance_cols is None:
        df0 = datasets[0][0]
        performance_cols = [
            c for c in df0.columns
            if any(kw in c.lower() for kw in ["rmse", "bias"])
        ]
        if not performance_cols:
            raise ValueError(f"No performance columns auto-detected.\n"
                             f"Available columns: {list(df0.columns)}\n"
                             f"Pass performance_cols= explicitly.")
        print(f"Auto-detected performance columns: {performance_cols}")

    # --- build one df_region per dataset ---
    def build_df_region(df_metrics):
        records = []
        for full_key in df_metrics.index:
            shift_key = f"{complement_key}{full_key}" if complement_key else full_key
            if shift_key not in all_shifts:
                print(f"Warning: '{shift_key}' not in all_shifts, skipping.")
                continue
            parts = shift_key.split("_TO_")
            src = parts[0].replace("XREG_", "")
            tgt = parts[1] if len(parts) > 1 else shift_key
            if tgt.upper() in exclude_targets or src.upper(
            ) in exclude_sources:
                continue
            shift = all_shifts[shift_key]
            row = {
                "full_key": full_key,
                "region": f"{src}→{tgt}",
                "src": src,
                "tgt": tgt
            }
            for pc in performance_cols:
                row[pc] = df_metrics.loc[full_key, pc]
            for dc in all_distance_cols:
                row[dc] = shift.get(dc, float("nan"))
            records.append(row)
        if not records:
            raise ValueError(f"No matching regions found.\n"
                             f"df_metrics index: {list(df_metrics.index)}\n"
                             f"all_shifts keys:  {list(all_shifts.keys())}")
        return pd.DataFrame(records)

    df_regions = [build_df_region(df) for df, _ in datasets]

    # --- shared color map across all datasets ---
    all_sources = sorted(
        {src
         for df in df_regions
         for src in df["src"].unique()})
    if color_palette is not None:
        src_colors = {
            s: color_palette.get(s, NATURE_COLORS[i % len(NATURE_COLORS)])
            for i, s in enumerate(all_sources)
        }
    else:
        src_colors = {s: c for s, c in zip(all_sources, NATURE_COLORS)}

    # --- layout: one column per dataset, one row per performance metric ---
    nrows = len(performance_cols)
    ncols = len(datasets)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize or nature_figsize(cols=ncols, height_mm=80 * nrows),
        squeeze=False,
        sharey="row" if sharey else False,
        sharex=False,
    )

    XLABEL_MAP = {
        "sinkhorn joint (true)": "Sinkhorn Distance (Joint)",
        "sinkhorn climate": "Sinkhorn Distance (Climate)",
        "sinkhorn topo": "Sinkhorn Distance (Topo)",
        "mmd2 joint (averaged)": "MMD² Distance (Joint)",
        "mmd2 climate": "MMD² Distance (Climate)",
        "mmd2 topo": "MMD² Distance (Topo)",
        "energy joint (averaged)": "Energy Distance (Joint)",
        "energy climate": "Energy Distance (Climate)",
        "energy topo": "Energy Distance (Topo)",
    }
    YLABEL_MAP = {
        "RMSE_annual": "Annual RMSE (m w.e.)",
        "RMSE_winter": "Winter RMSE (m w.e.)",
    }

    for col_idx, (df_region, panel_title) in enumerate(
            zip(df_regions, (t for _, t in datasets))):
        # only one distance col expected when using override; iterate for generality
        dc = distance_cols[
            0]  # with override typically one col → one panel per dataset

        for row_idx, pc in enumerate(performance_cols):
            ax = axes[row_idx][col_idx]

            x = df_region[dc].values.astype(float)
            y = df_region[pc].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)

            for _, row in df_region.iterrows():
                xi, yi = float(row[dc]), float(row[pc])
                if not (np.isfinite(xi) and np.isfinite(yi)):
                    continue
                ax.scatter(xi,
                           yi,
                           color=src_colors[row["src"]],
                           s=80,
                           zorder=3,
                           edgecolors="white",
                           linewidths=0.4)
                x_range = x[mask].max() - x[mask].min() if mask.sum(
                ) > 1 else 1
                x_frac = (xi - x[mask].min()) / x_range if x_range > 0 else 0
                xytext, ha = ((-6, 4), "right") if x_frac > 0.7 else ((6, 4),
                                                                      "left")
                ax.annotate(
                    row["region"],
                    (xi, yi),
                    xytext=xytext,
                    textcoords="offset points",
                    fontsize=NATURE_SPECS["font_min_pt"],
                    fontweight="bold",
                    color=src_colors[row["src"]],
                    ha=ha,
                )

            n_valid = mask.sum()
            if n_valid >= 3:
                rho, pval = stats.spearmanr(x[mask], y[mask])
                corr_txt = (
                    f"rho = {rho:.2f}  blur = {blur_map[dc]:.3f}  n = {n_valid}"
                    if dc in blur_map else
                    f"rho = {rho:.2f}  p = {pval:.2f}  n = {n_valid}")
                slope, intercept, *_ = stats.linregress(x[mask], y[mask])
                x_line = np.linspace(x[mask].min(), x[mask].max(), 100)
                ax.plot(x_line,
                        slope * x_line + intercept,
                        color="black",
                        linewidth=1.2,
                        linestyle="--",
                        alpha=0.3,
                        zorder=2)
            else:
                corr_txt = f"n = {n_valid} (too few for rho)"

            ax.text(0.04,
                    0.97,
                    corr_txt,
                    transform=ax.transAxes,
                    va="top",
                    ha="left",
                    fontsize=NATURE_SPECS["font_min_pt"],
                    bbox=dict(boxstyle="round,pad=0.3",
                              fc="white",
                              ec="none",
                              alpha=0.8))

            apply_nature_style(ax,
                               fontsize=NATURE_SPECS["font_min_pt"],
                               box=True)
            ax.set_xlabel(
                XLABEL_MAP.get(xlabel_map.get(dc, dc), xlabel_map.get(dc, dc)),
                fontsize=NATURE_SPECS["font_min_pt"],
            )
            # only label y-axis on the leftmost column
            if col_idx == 0:
                ax.set_ylabel(YLABEL_MAP.get(pc, pc),
                              fontsize=NATURE_SPECS["font_min_pt"])

            if row_idx == 0:
                ax.set_title(panel_title, fontsize=NATURE_SPECS["font_max_pt"])

    # --- shared legend ---
    legend_handles = [
        plt.scatter([], [], color=src_colors[s], s=80, label=s)
        for s in all_sources
    ]
    fig.legend(
        handles=legend_handles,
        title="Source region",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.04),
        ncols=len(all_sources),
        frameon=False,
    )

    fig.suptitle(
        "Region-level domain shift vs transfer performance",
        fontsize=NATURE_SPECS["font_max_pt"] + 1,
        y=1.01,
    )
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    return fig, df_regions

In [ ]:
fig, (df_zs, df_ft) = plot_region_shift_vs_performance_multi(
    datasets=[
        (df_metrics_zs_all, f"Zero-shot ({SPLIT})"),
        (df_metrics_ft_all, f"Fine-tuned adapter ({SPLIT})"),
    ],
    all_shifts=all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
    complement_key="",
    distance_cols_override=["D_sinkhorn_topo"],
    color_palette=color_palette)
plt.show()

In [ ]:
df_metrics_zs_all = _extract_metrics_for_strategy("no_ft")
df_metrics_ft_all = _extract_metrics_for_strategy("dan")

# --- combine shifts ---
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source.get(src_region, {}))

# --- apply exclusions ---
if EXCLUDE_TARGETS:
    excl_pattern = "|".join(EXCLUDE_TARGETS)
    df_metrics_zs_all = df_metrics_zs_all[~df_metrics_zs_all.index.str.
                                          contains(excl_pattern)]
    df_metrics_ft_all = df_metrics_ft_all[~df_metrics_ft_all.index.str.
                                          contains(excl_pattern)]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

print(
    f"Zero-shot metrics: {df_metrics_zs_all.shape} — {list(df_metrics_zs_all.index)}"
)
print(
    f"FT metrics:        {df_metrics_ft_all.shape} — {list(df_metrics_ft_all.index)}"
)